In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [3]:
filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot.pkl'
with open(filename, "rb") as f:
    dict_uniprot = pickle.load(f)

In [6]:
filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot_new.pkl'
with open(filename, "rb") as f:
    dict_dn_DMS = pickle.load(f)

In [9]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [8]:
len(dict_dn_DMS.keys())

428

In [5]:
len(dict_uniprot.keys())

428

In [ ]:
def make_mutation_fitness(domain_id, df):

    domain_id_list=domain_id.split("_")
    dom_pos = float(domain_id_list[-1])

    df_one_protein = df.where(df['domain_ID'] == domain_id).dropna()

    dom_position = df_one_protein['position'] - dom_pos
    df_one_protein.insert(loc=0, column='real_position', value=dom_position)
    df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

    df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)
    
    df_mutation = df_one_protein_ns[['wt_seq','real_position','mut_aa','normalized_fitness']]
    

In [10]:
DMS_keys = list(dict_dn_DMS.keys())

In [12]:
i = 1
key = DMS_keys[i]

key

'O94929'

In [13]:
dict_dn_DMS[key]

{'sequence': 'MNTSIPYQQNPYNPRGSSNVIQCYRCGDTCKGEVVRVHNNHFHIRCFTCQVCGCGLAQSGFFFKNQEYICTQDYQQLYGTRCDSCRDFITGEVISALGRTYHPKCFVCSLCRKPFPIGDKVTFSGKECVCQTCSQSMASSKPIKIRGPSHCAGCKEEIKHGQSLLALDKQWHVSCFKCQTCSVILTGEYISKDGVPYCESDYHAQFGIKCETCDRYISGRVLEAGGKHYHPTCARCVRCHQMFTEGEEMYLTGSEVWHPICKQAARAEKKLKHRRTSETSISPPGSSIGSPNRVICAKVDNEILNYKDLAALPKVKSIYEVQRPDLISYEPHSRYMSDEMLERCGYGESLGTLSPYSQDIYENLDLRQRRASSPGYIDSPTYSRQGMSPTFSRSPHHYYRSGPESGRSSPYHSQLDVRSSTPTSYQAPKHFHIPAGDSNIYRKPPIYKRHGDLSTATKSKTSEDISQTSKYSPIYSPDPYYASESEYWTYHGSPKVPRARRFSSGGEEDDFDRSMHKLQSGIGRLILKEEMKARSSSYADPWTPPRSSTSSREALHTAGYEMSLNGSPRSHYLADSDPLISKSASLPAYRRNGLHRTPSADLFHYDSMNAVNWGMREYKIYPYELLLVTTRGRNRLPKDVDRTRLERHLSQEEFYQVFGMTISEFDRLALWKRNELKKQARLF',
 'pfam': 'PF00412'}

In [14]:
dict_uniprot[key]

'MNTSIPYQQNPYNPRGSSNVIQCYRCGDTCKGEVVRVHNNHFHIRCFTCQVCGCGLAQSGFFFKNQEYICTQDYQQLYGTRCDSCRDFITGEVISALGRTYHPKCFVCSLCRKPFPIGDKVTFSGKECVCQTCSQSMASSKPIKIRGPSHCAGCKEEIKHGQSLLALDKQWHVSCFKCQTCSVILTGEYISKDGVPYCESDYHAQFGIKCETCDRYISGRVLEAGGKHYHPTCARCVRCHQMFTEGEEMYLTGSEVWHPICKQAARAEKKLKHRRTSETSISPPGSSIGSPNRVICAKVDNEILNYKDLAALPKVKSIYEVQRPDLISYEPHSRYMSDEMLERCGYGESLGTLSPYSQDIYENLDLRQRRASSPGYIDSPTYSRQGMSPTFSRSPHHYYRSGPESGRSSPYHSQLDVRSSTPTSYQAPKHFHIPAGDSNIYRKPPIYKRHGDLSTATKSKTSEDISQTSKYSPIYSPDPYYASESEYWTYHGSPKVPRARRFSSGGEEDDFDRSMHKLQSGIGRLILKEEMKARSSSYADPWTPPRSSTSSREALHTAGYEMSLNGSPRSHYLADSDPLISKSASLPAYRRNGLHRTPSADLFHYDSMNAVNWGMREYKIYPYELLLVTTRGRNRLPKDVDRTRLERHLSQEEFYQVFGMTISEFDRLALWKRNELKKQARLF'